## 3. Data Pipeline

In [ ]:
import pandas as pd
from collections import defaultdict
import imagehash

### 3.2.3 De-dup
Duplicate and near-duplicate images were identified and removed to reduce redundancy and prevent data leakage between training and evaluation sets.


**Exact Duplicate Removal**
- Perceptual hash (pHash) values were used to identify exact duplicate images.
- Images sharing the same pHash were considered visually identical.
- For each group of exact duplicates, only the image with the highest resolution (largest number of pixels) was retained.
- Lower-resolution duplicates were removed from the dataset.

In [12]:
df = pd.read_csv("mushroom_paths.csv")

Exact duplicates

This removes images that have the same perceptual hash and keeps the one with largest num_pixels (your step (b) and (c)).

In [14]:
df_sorted = df.sort_values(["phash", "num_pixels"], ascending=[True, False])

dedup = df_sorted.drop_duplicates(subset=["phash"], keep="first").copy()

removed = len(df) - len(dedup)
print(f"Rows before: {len(df):,}")
print(f"Rows after : {len(dedup):,}")
print(f"Removed    : {removed:,} ({removed/len(df)*100:.2f}%)")

dedup.to_csv("mushroomDedupExact.csv", index=False)
print("Saved:", "mushroomDedupExact.csv")

Rows before: 102,711
Rows after : 94,698
Removed    : 8,013 (7.80%)
Saved: mushroomDedupExact.csv


**Near-Duplicate Removal**
- Near-duplicate images were identified by computing the Hamming distance between pHash values.
- Images with a Hamming distance below a predefined threshold were considered near-duplicates.
- When near-duplicate images were detected, the image with the highest resolution was retained, and remaining lower-resolution images were removed.

In [ ]:
HAMMING_THRESHOLD = 5     # 5 means "very similar"
PREFIX_LEN = 4            # bucket by first 4 hex chars; increase to reduce comparisons

In [ ]:
# Convert to ImageHash
hashes = df["phash"].astype(str).apply(imagehash.hex_to_hash)

In [17]:
# Bucket indices by hash prefix to avoid O(N^2)
buckets = defaultdict(list)
for i, h in enumerate(hashes):
    buckets[str(h)[:PREFIX_LEN]].append(i)

keep = set(range(len(df)))
dropped = set()

In [ ]:
# Within each bucket, drop near-duplicates by keeping highest resolution
for _, idxs in buckets.items():
    if len(idxs) <= 1:
        continue

    idxs_sorted = sorted(idxs, key=lambda i: df.loc[i, "num_pixels"], reverse=True)

    chosen = []
    for i in idxs_sorted:
        if i in dropped:
            continue
        is_near_dup = any((hashes[i] - hashes[j]) <= HAMMING_THRESHOLD for j in chosen)
        if is_near_dup:
            dropped.add(i)
        else:
            chosen.append(i)

In [19]:
keep = sorted(list(keep - dropped))
out = df.iloc[keep].copy()

print(f"Rows before: {len(df):,}")
print(f"Rows after : {len(out):,}")
print(f"Removed    : {len(df)-len(out):,} ({(len(df)-len(out))/len(df)*100:.2f}%)")

out.to_csv("mushroomDedupNear.csv", index=False)
print("Saved:", "mushroomDedupNear.csv")

Rows before: 102,711
Rows after : 94,631
Removed    : 8,080 (7.87%)
Saved: mushroomDedupNear.csv


### 3.2.4 Standardization and Augmentation

**Standardization**

Standardization was applied dynamically during data loading to ensure consistent input format for model training. All images were resized to a fixed input resolution to accommodate convolutional neural network requirements. Pixel values were scaled to the range [0, 1] and normalized using predefined channel-wise mean and standard deviation values. This process was deterministic and applied consistently across training, validation, and test datasets.

**Augmentation**

Data augmentation was applied only to the training dataset to improve model robustness and generalization. Random transformations, including horizontal flipping, rotation, and color jittering, were used to introduce controlled variability in image appearance while preserving semantic content. Augmentation was performed at load time and did not modify the original image files.

In [20]:
from torchvision import transforms

In [45]:
IMG_SIZE = 224

# If using pretrained ImageNet models, use ImageNet mean/std:
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

train_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    transforms.ToTensor(),  # converts to [0,1]
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

val_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])


Dataset class was implemented to load images and labels from the metadata CSV and apply preprocessing transforms dynamically during training.

In [47]:
from PIL import Image
from torch.utils.data import Dataset

In [48]:
class MushroomDataset(Dataset):
    def __init__(self, csv_path, transform=None, label_col="class"):
        self.df = pd.read_csv(csv_path)
        self.transform = transform

        # Encode labels
        self.label_col = label_col
        self.classes = sorted(self.df[label_col].unique().tolist())
        self.class_to_idx = {c:i for i,c in enumerate(self.classes)}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["file_path"]).convert("RGB")
        y = self.class_to_idx[row[self.label_col]]

        if self.transform:
            img = self.transform(img)

        return img, y
